# 10 · Inspect, measure, and operate Kafka

**Level:** advanced · **Time:** 55–70 minutes · **Prerequisites:** notebooks 01–09.

Inspect real topic metadata and configuration, measure committed consumer lag, compare
batching and compression, and rebuild a keyed view with a tombstone. Finish by mapping
the local lab to authenticated production connections and realistic failure drills.

Start Kafka, restart the kernel, and run top to bottom. This is a **single-broker local
lab**: it cannot demonstrate replica failover or durable operation after broker loss.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
import json
import os
import time
from confluent_kafka import KafkaError, KafkaException, TopicPartition
from confluent_kafka.admin import AdminClient, ConfigResource, ResourceType

require_broker()
admin = AdminClient(client_config())
observed_topic = new_topic("10.observe", partitions=3)


## 1. Inspect leaders, replicas, and ISR

A partition's leader serves its writes. Replicas hold copies, and ISR is the set of
replicas currently considered in sync. One local replica explains why `acks=all` here
still means just one broker acknowledged the record.


In [ ]:
metadata = admin.list_topics(topic=observed_topic, timeout=10)
for partition, info in sorted(metadata.topics[observed_topic].partitions.items()):
    print({"partition": partition, "leader": info.leader,
           "replicas": info.replicas, "isr": info.isrs})
assert len(metadata.topics[observed_topic].partitions) == 3


In [ ]:
resource = ConfigResource(ResourceType.TOPIC, observed_topic)
entries = admin.describe_configs([resource], request_timeout=10)[resource].result(timeout=15)
for name in ["cleanup.policy", "retention.ms", "segment.bytes", "min.insync.replicas"]:
    entry = entries[name]
    print(name, "=", entry.value, "| source:", entry.source)


## 2. Measure committed lag independently from the current position

For an ordinary nontransactional topic, a useful approximation is
`log-end offset − committed next offset`, per partition. A consumer's current position
may be ahead of its committed offset. A negative committed offset is a sentinel for
“no committed offset”, not a real negative record number.

This is offset lag, not necessarily the exact count of application records: compaction,
retention, and transaction control records can create gaps. Transactional visibility also
has a last-stable-offset boundary. Lag in seconds needs event-age measurements too.


In [ ]:
lag_topic = new_topic("10.lag", partitions=1)
publish(lag_topic, [{"event_id": str(i)} for i in range(10)])
lag_group = f"lab-10-lag-{uuid4().hex}"
lag_reader = consumer(lag_group)
try:
    lag_reader.subscribe([lag_topic])
    first_four = poll_records(lag_reader, count=4, timeout=20)
    lag_reader.commit(message=first_four[-1], asynchronous=False)
    next_two = poll_records(lag_reader, count=2, timeout=10)
    partition = TopicPartition(lag_topic, 0)
    low, high = lag_reader.get_watermark_offsets(partition, timeout=10, cached=False)
    committed_offset = lag_reader.committed([partition], timeout=10)[0].offset
    position = lag_reader.position([partition])[0].offset
    print({"low": low, "high": high, "committed": committed_offset,
           "position": position, "committed_lag": high - committed_offset})
    assert (low, high, committed_offset, position) == (0, 10, 4, 6)
finally:
    lag_reader.close()


In [ ]:
remaining = collect(lag_topic, count=6, group_id=lag_group, commit=True)
assert [decode(m)["event_id"] for m in remaining] == [str(i) for i in range(4, 10)]
print("Restart began at the committed offset 4, replaying the two uncommitted reads.")


## 3. Benchmark delivered records, not enqueue calls

`produce()` normally enqueues work locally. Timing only that call measures queueing,
not delivery. The small experiment below includes flush and successful delivery reports.
We keep `acks=all` and idempotence enabled in every variant so reliability settings do
not confound the comparison.

Connection warm-up, CPU, Docker resource limits, message size, and noise dominate small
runs. This is a measurement exercise, not a production sizing result or a promise that
compression always helps. Compression effectiveness also depends on payload entropy.


In [ ]:
N = 500
payloads = [json.dumps({"event_id": str(i), "category": "demo", "body": "abc" * 100}).encode()
            for i in range(N)]

def benchmark(label, overrides):
    topic = new_topic(f"10.bench.{label}", partitions=3)
    p = producer(**overrides)
    # Warm metadata, so the timed region is less dominated by topic discovery.
    p.list_topics(topic=topic, timeout=10)
    delivered, errors = [], []
    def report(error, message):
        if error:
            errors.append(str(error))
        else:
            delivered.append(message.offset())
    started = time.perf_counter()
    try:
        for i, payload in enumerate(payloads):
            p.produce(topic, key=str(i % 10), value=payload, on_delivery=report)
            p.poll(0)
    finally:
        pending = p.flush(15)
    elapsed = time.perf_counter() - started
    assert not errors and pending == 0 and len(delivered) == N, (errors, pending)
    return {"variant": label, "records": N, "seconds": round(elapsed, 4),
            "delivered_per_second": round(N / elapsed),
            "application_bytes": sum(map(len, payloads))}


In [ ]:
measurements = [
    benchmark("immediate", {"linger.ms": 0, "compression.type": "none"}),
    benchmark("batched", {"linger.ms": 20, "compression.type": "none"}),
    benchmark("batched-gzip", {"linger.ms": 20, "compression.type": "gzip"}),
]
for row in measurements:
    print(row)


**Interpretation:** `application_bytes` is the uncompressed payload size, not measured
network bytes. Batching can increase throughput while adding latency; gzip spends CPU
to reduce suitable payloads. Repeat variants in shuffled order, separate warm-up, record
latency percentiles and resource use, and include a consumer before making decisions.


## 4. Build a keyed view and delete a key with a tombstone

On a compacted topic, the key identifies the entity. A tombstone is a **null Kafka value**,
not the JSON string `"null"`. Compaction eventually removes obsolete versions; it does
not deduplicate records immediately or promise one record per key in a fresh read.


In [ ]:
compacted_topic = new_topic("10.customer-view", partitions=1,
                            config={"cleanup.policy": "compact"})
compact_producer = producer()
compact_errors = []
def compact_report(error, message):
    if error:
        compact_errors.append(str(error))
changes = [
    ("c1", {"name": "Asha", "tier": "basic"}),
    ("c1", {"name": "Asha", "tier": "gold"}),
    ("c2", {"name": "Ravi", "tier": "basic"}),
    ("c2", None),
]
for key, value in changes:
    compact_producer.produce(compacted_topic, key=key,
        value=None if value is None else json.dumps(value).encode(),
        on_delivery=compact_report)
    compact_producer.poll(0)
assert compact_producer.flush(15) == 0 and not compact_errors


In [ ]:
# Capture an offset boundary, not an expected number of retained records.
# Compaction may have removed older values while preserving their offset gaps.
snapshot_reader = consumer(f"lab-10-snapshot-{uuid4().hex}",
                           **{"enable.partition.eof": True})
changes_read = []
try:
    partition = TopicPartition(compacted_topic, 0)
    low, snapshot_end = snapshot_reader.get_watermark_offsets(partition, timeout=10)
    snapshot_reader.assign([TopicPartition(compacted_topic, 0, low)])
    reached_end = low >= snapshot_end
    deadline = time.monotonic() + 20
    while not reached_end and time.monotonic() < deadline:
        message = snapshot_reader.poll(0.5)
        if message is None:
            continue
        if message.error():
            if message.error().code() == KafkaError._PARTITION_EOF:
                reached_end = message.offset() >= snapshot_end
                continue
            raise KafkaException(message.error())
        if message.offset() < snapshot_end:
            changes_read.append(message)
        reached_end = message.offset() + 1 >= snapshot_end
    if not reached_end:
        raise TimeoutError("Did not reach the captured compacted-topic end offset.")
finally:
    snapshot_reader.close()

view = {}
for message in changes_read:
    key = message.key().decode()
    if message.value() is None:
        view.pop(key, None)
    else:
        view[key] = decode(message)
assert view == {"c1": {"name": "Asha", "tier": "gold"}}
print("Retained changes read:", len(changes_read), "| captured end offset:", snapshot_end)
print("Materialized view:", view)


The read stops at the end offset captured before polling. It can read all four records,
or fewer if cleaning has already removed obsolete versions; both reconstruct the same
current keyed view in this isolated example. An EOF event also lets the scan finish when
offsets have gaps. This bound is not database-style snapshot isolation against concurrent
writers or cleaning. Do not depend on a fixed cleaning delay or an exact record count. Tombstones
can eventually expire; a slow or rebuilding consumer must respect the deletion-retention
window. Combining compaction with time deletion changes rebuilding assumptions too.


## 5. Inspect connection security without printing credentials

The bundled Docker broker uses loopback plaintext for local learning. Authentication,
TLS, and authorization must be configured on the broker or managed service as well as
on clients. Client settings alone cannot secure a plaintext broker.

The helper reads `KAFKA_SECURITY_PROTOCOL`, `KAFKA_SASL_MECHANISM`,
`KAFKA_SASL_USERNAME`, `KAFKA_SASL_PASSWORD`, and `KAFKA_SSL_CA_LOCATION` from
the environment or local `.env`. Keep secret values out of notebook cells and outputs.


In [ ]:
connection = client_config()
print("Security protocol:", connection.get("security.protocol", "PLAINTEXT"))
print("SASL mechanism configured:", "sasl.mechanism" in connection)
print("Username configured:", bool(connection.get("sasl.username")))
print("Password configured:", bool(connection.get("sasl.password")))
print("Custom CA configured:", bool(connection.get("ssl.ca.location")))
# Do not display the full configuration dictionary: it may contain secrets.


For an authorized SASL/SCRAM cluster, obtain the bootstrap address, CA requirements,
and credentials from its administrator. Set `KAFKA_SECURITY_PROTOCOL=SASL_SSL` and
the required mechanism, for example `SCRAM-SHA-512`, in your private environment.
Then restart the kernel and run the preflight. Keep certificate verification enabled.
Topic creation and transactions may need additional ACLs; a production application
identity often should not have general administration permissions.

This section is a configuration walkthrough, not an authenticated-broker exercise.
Do not point these topic-creating notebooks at a shared production cluster.


## 6. An operations drill you can explain and repeat

In a terminal, `docker compose stop kafka` makes the local broker unavailable. Run
`require_broker(timeout=3)` in a separate scratch cell and observe the bounded failure.
Restart with `docker compose start kafka`, wait for health, then retry preflight. Do not
execute this stop command while other notebooks or Django workers are using the broker.

This is an outage-and-recovery drill, **not replica failover**. A real high-availability
drill needs multiple brokers/controllers, replication and minimum-ISR settings, a measured
leader election, and verification of writes before and after failure. Increasing the
replication factor to 3 on this one-broker lab cannot create that capacity.


## Practice

1. Consume five records but commit only two; calculate both position lag and committed lag.
2. Replace the repetitive benchmark body with random bytes and compare gzip again.
3. Publish another `c1` update after the tombstone for `c2`; rebuild the keyed view.
4. Write an alert specification for stalled progress, growing event age, delivery errors,
   disk pressure, unavailable partitions, and under-replicated partitions.

**Solution hints:** on ten records the two lags are 5 and 8; incompressible data can cost
CPU with little bandwidth saving; replay applies the latest value by key; each alert needs
a threshold, duration, owner, and useful remediation steps.


## Presenter checkpoint and references

Explain why low lag does not prove correct business processing, why `acks=all` cannot
compensate for a single replica, and why compaction is not immediate deduplication.

[Python AdminClient and offsets](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html),
[Kafka monitoring](https://kafka.apache.org/41/operations/monitoring/),
[Kafka log compaction](https://kafka.apache.org/41/design/design/#log-compaction),
[Kafka security](https://kafka.apache.org/41/security/overview/).
